**1. Import Libraries and Mount to Drive**

In [ ]:
!pip install evaluate -q
!pip install --upgrade torchao -q

# import libraries
import zipfile
import os
import torch
import re
import evaluate
import random
import math
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

# mount to personal google drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.7 MB/s eta 0:00:00


Mounted at /content/drive


**2. Load Model Adapters From Google Drive**

In [ ]:
# path to the model zip file in google drive
zip_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Low-Rank-Adaptation-(LoRA)/Llama-3.2-3B-Instruct/Medical-Math/medical-math_llama3_lora_adapter.zip"

# where to unzip the model in colab
output_path = "/content/model"

# unzips the model into the colab workspace
print("Unzipping model...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_path)

# confirms the model has been unzipped and shows the folder contents
print(f"Model unzipped to {output_path}")
print(os.listdir(output_path))

Unzipping model...
Model unzipped to /content/model
['medical-math_llama3_lora_adapter']


**3. Load Model Llama-3.2-3B-Instruct and apply adapters (Medical + Math)**


In [ ]:
# path to the unzipped lora adapter
adapter_path = "/content/model/medical-math_llama3_lora_adapter"

# base model name from hugging face
model_name = "meta-llama/Llama-3.2-3B-Instruct"

# loads the tokenizer from the base model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# loads the base model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

# loads the lora adapter on top of the base model
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

# check if the lora adapter loaded correctly
if isinstance(model, PeftModel):
    print("LoRA adapter is active!")
else:
    print("Warning: adapter did not load, running base model only.")

print("Model loaded and ready!")

Loading tokenizer...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loading LoRA adapter...
LoRA adapter is active!
Model loaded and ready!


**4. Load compute_overall_accuracy**

In [ ]:
# official accuracy computation function from medcalc-bench (table_stats.py)
def compute_overall_accuracy(output_path, model_name, prompt_style):
    category_accuracy = {}

    with open(f"outputs/{output_path}") as file:
        for line in file:
            data = json.loads(line)

            category = data["Category"]

            if category not in category_accuracy:
                category_accuracy[category] = []

            if data["Result"] == "Correct":
                category_accuracy[category].append(1)
            else:
                category_accuracy[category].append(0)

    # compute average and standard deviation for each category
    category_stats = {}
    all_results = []

    for cat, results in category_accuracy.items():
        results_array = np.array(results)
        category_mean = np.mean(results_array)
        category_std = round(np.sqrt(category_mean * (1-category_mean) / len(results_array)), 2)
        category_stats[cat] = {
            "average": round(category_mean * 100, 2),
            "std": category_std
        }
        all_results.extend(results)

    # compute overall average and standard deviation
    all_results_array = np.array(all_results)
    overall_average = np.mean(all_results_array)
    overall_std =  round(np.sqrt(overall_average * (1-overall_average) / 1047), 2)

    category_stats["overall"] = {
        "average": round(overall_average * 100, 2),
        "std": overall_std
    }

    if not os.path.exists("results"):
        os.makedirs("results")

    if "/" in model_name:
        model_name = model_name.split('/')[1]

    with open(f"results/results_{model_name}_{prompt_style}.json", "w") as file:
        json.dump(category_stats, file, indent=4)

    return category_stats

**5. Perform Evaluation (MedCalc-Bench-v1.0)**

In [ ]:
# set device to gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# set seed for reproducibility
torch.manual_seed(42)

# official direct answer prompt from medcalc-bench
def direct_answer(note, question):
    system_msg = 'You are a helpful assistant for calculating a score for a given patient note. Please output answer only without any other text. Your output should only contain a JSON dict formatted as {"answer": str(value which is the answer to the question)}.'
    user_temp = f'Here is the patient note:\n{note}\n\nHere is the task:\n{question}\n\nPlease directly output the JSON dict formatted as {{"answer": str(value which is the answer to the question)}}:'
    return system_msg, user_temp

# official extract answer function from medcalc-bench
def extract_answer(answer, calid):
    calid = int(calid)
    extracted_answer = re.findall(r'[Aa]nswer":\s*(.*?)\}', answer)

    # tries to also grab the reasoning if it exists
    matches = re.findall(r'"step_by_step_thinking":\s*"([^"]+)"\s*,\s*"[Aa]nswer"', answer)
    if matches:
        explanation = matches[-1]
    else:
        explanation = "No Explanation"

    # checks if an answer was found
    if len(extracted_answer) == 0:
        extracted_answer = "Not Found"
    else:
        extracted_answer = extracted_answer[-1].strip().strip('"')
        if extracted_answer in [
            "str(short_and_direct_answer_of_the_question)",
            "str(value which is the answer to the question)",
            "X.XX"
        ]:
            extracted_answer = "Not Found"

    if calid in [13, 68]:
        match = re.search(r"^(0?[1-9]|1[0-2])\/(0?[1-9]|[12][0-9]|3[01])\/(\d{4})", extracted_answer)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            year = match.group(3)
            answer = f"{month:02}/{day:02}/{year}"
        else:
            answer = "N/A"

    elif calid in [69]:
        extracted_answer = extracted_answer.replace("[", "(").replace("]", ")").replace("'", "").replace('"', "")
        match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", extracted_answer)
        if match:
            weeks = match.group(1)
            days = match.group(3)
            answer = f"({weeks}, {days})"
        else:
            answer = "N/A"

    elif calid in [4, 15, 16, 17, 18, 20, 21, 25, 27, 28, 29, 32, 33, 36, 43, 45, 48, 51]:
        match = re.search(r"(\d+) out of", extracted_answer)
        if match:
            answer = match.group(1)
        else:
            match = re.search(r"-?\d+(, ?-?\d+)+", extracted_answer)
            if match:
                answer = str(len(match.group(0).split(",")))
            else:
                match = re.findall(r"(-?\d+(\.\d+)?)", extracted_answer)
                if len(match) > 0:
                    answer = match[-1][0]
                else:
                    answer = "N/A"

    elif calid in [2, 3, 5, 6, 7, 8, 9, 10, 11, 19, 22, 23, 24, 26, 30, 31, 38, 39, 40, 44, 46, 49, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]:
        match = re.search(r"str\((.*)\)", extracted_answer)
        if match:
            expression = match.group(1).replace("^", "**").replace("is odd", "% 2 == 1").replace("is even", "% 2 == 0").replace("sqrt", "math.sqrt").replace(".math", "").replace("weight", "").replace("height", "").replace("mg/dl", "").replace("g/dl", "").replace("mmol/L", "").replace("kg", "").replace("g", "").replace("mEq/L", "")
            expression = expression.split('#')[0]
            if expression.count('(') > expression.count(')'):
                expression += ')' * (expression.count('(') - expression.count(')'))
            elif expression.count(')') > expression.count('('):
                expression = '(' * (expression.count(')') - expression.count('(')) + expression
            try:
                answer = eval(expression, {"__builtins__": None}, {"min": min, "pow": pow, "round": round, "abs": abs, "int": int, "float": float, "math": math, "np": np, "numpy": np})
            except:
                answer = "N/A"
        else:
            match = re.search(r"(-?\d+(\.\d+)?)\s*mL/min/1.73", extracted_answer)
            if match:
                answer = eval(match.group(1))
            else:
                match = re.findall(r"(-?\d+(\.\d+)?)\%", extracted_answer)
                if len(match) > 0:
                    answer = eval(match[-1][0]) / 100
                else:
                    match = re.findall(r"(-?\d+(\.\d+)?)", extracted_answer)
                    if len(match) > 0:
                        answer = eval(match[-1][0])
                    else:
                        answer = "N/A"
        if answer != "N/A":
            answer = str(answer)
    else:
        answer = "N/A"

    return answer, explanation

# official correctness check from medcalc-bench
def check_correctness(answer, ground_truth, calid, upper_limit, lower_limit):
    calid = int(calid)
    try:
        if calid in [13, 68]:
            if datetime.strptime(answer, "%m/%d/%Y").strftime("%-m/%-d/%Y") == datetime.strptime(ground_truth, "%m/%d/%Y").strftime("%-m/%-d/%Y"):
                correctness = 1
            else:
                correctness = 0

        elif calid in [69]:
            match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", ground_truth)
            ground_truth = f"({match.group(1)}, {match.group(3)})"
            match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", answer)
            if match:
                weeks = match.group(1)
                days = match.group(3)
                answer = f"({weeks}, {days})"
                correctness = 1 if eval(answer) == eval(ground_truth) else 0
            else:
                correctness = 0

        elif calid in [4, 15, 16, 17, 18, 20, 21, 25, 27, 28, 29, 32, 33, 36, 43, 45, 48, 51]:
            answer = round(eval(answer))
            correctness = 1 if answer == eval(ground_truth) else 0

        elif calid in [2, 3, 5, 6, 7, 8, 9, 10, 11, 19, 22, 23, 24, 26, 30, 31, 38, 39, 40, 44, 46, 49, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]:
            answer = eval(answer)
            correctness = 1 if eval(lower_limit) <= answer <= eval(upper_limit) else 0

        else:
            return None

    except Exception:
        return None

    return correctness

# load the official test set
df = pd.read_csv("test_data.csv")
test_set = df

print(f"Running evaluation on all {len(test_set)} questions...")

correct_count = 0
wrong_count = 0
skipped_count = 0
total = len(test_set)

category_stats = {}

output_path = f"{model_name.replace('/', '_')}_direct_answer.jsonl"

if not os.path.exists("outputs"):
    os.makedirs("outputs")

# set eos tokens based on model family
if "llama" in model_name.lower():
    eos_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]
elif "qwen" in model_name.lower():
    eos_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]
else:
    eos_token_ids = [tokenizer.eos_token_id]

# official loop from medcalc-bench
for index in tqdm(range(len(test_set))):
    example = test_set.iloc[index]

    system, user = direct_answer(example["Patient Note"], example["Question"])

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False,
            eos_token_id=eos_token_ids,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    try:
        predicted, explanation = extract_answer(response, example['Calculator ID'])
    except Exception as e:
        predicted = "N/A"
        explanation = str(e)

    if predicted not in ["N/A", "Not Found", None]:
        try:
            correctness = check_correctness(
                answer=predicted,
                ground_truth=example['Ground Truth Answer'],
                calid=example['Calculator ID'],
                upper_limit=example['Upper Limit'],
                lower_limit=example['Lower Limit']
            )
        except Exception:
            correctness = None
    else:
        correctness = None

    if correctness is None:
        skipped_count += 1
        result_str = "Skipped"
    elif correctness == 1:
        correct_count += 1
        result_str = "Correct"
    else:
        wrong_count += 1
        result_str = "Wrong"

    cat = example.get('Category', 'Unknown')
    if cat not in category_stats:
        category_stats[cat] = {'correct': 0, 'wrong': 0, 'skipped': 0}
    if correctness is None:
        category_stats[cat]['skipped'] += 1
    elif correctness == 1:
        category_stats[cat]['correct'] += 1
    else:
        category_stats[cat]['wrong'] += 1

    print(f"\n{'-' * 50}")
    print(f"Question {index+1} of {total}")
    print(f"Calculator:       {example['Calculator Name']}")
    print(f"Output type:      {example['Output Type']}")
    print(f"Ground truth:     {example['Ground Truth Answer']}")
    print(f"Acceptable range: {example['Lower Limit']} - {example['Upper Limit']}")
    print(f"Model response:   {response}")
    print(f"Extracted answer: {predicted}")
    print(f"Result:           {result_str}")
    print(f"{'-' * 50}")

    with open(f"outputs/{output_path}", "a") as f:
        f.write(json.dumps({
            "Row Number": int(example["Row Number"]),
            "Calculator Name": example['Calculator Name'],
            "Calculator ID": str(example['Calculator ID']),
            "Category": example['Category'],
            "Note ID": str(example['Note ID']),
            "Patient Note": example['Patient Note'],
            "Question": example['Question'],
            "LLM Answer": predicted,
            "LLM Explanation": "N/A",
            "Ground Truth Answer": example['Ground Truth Answer'],
            "Ground Truth Explanation": example['Ground Truth Explanation'],
            "Result": "Correct" if correctness == 1 else "Incorrect"
        }) + "\n")

# calculates the final stats
answered = correct_count + wrong_count
coverage = (answered / total * 100)
accuracy_all = (correct_count / total * 100)

print("\n" + "=" * 50)
print("EVALUATION COMPLETE")
print(f"Total questions:          {total}")
print(f"Correct answers:          {correct_count}")
print(f"Wrong answers:            {wrong_count}")
print(f"Skipped/Invalid:          {skipped_count}")
print(f"Coverage:                 {coverage:.2f}%")
print(f"Accuracy (all {total}):   {accuracy_all:.2f}%")
print("=" * 50)

# official accuracy checker
compute_overall_accuracy(output_path, model_name, "direct_answer")

Running evaluation on all 1047 questions...



  0%|          | 0/1047 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

  0%|          | 1/1047 [00:01<32:35,  1.87s/it]


--------------------------------------------------
Question 1 of 1047
Calculator:       Creatinine Clearance (Cockcroft-Gault Equation)
Output type:      decimal
Ground truth:     25.238
Acceptable range: 23.9761 - 26.4999
Model response:   {"answer": "24.0"}
Extracted answer: 24.0
Result:           Correct
--------------------------------------------------



  0%|          | 2/1047 [00:02<20:56,  1.20s/it]


--------------------------------------------------
Question 2 of 1047
Calculator:       Creatinine Clearance (Cockcroft-Gault Equation)
Output type:      decimal
Ground truth:     38
Acceptable range: 36.1 - 39.9
Model response:   {"answer": "24.6"}
Extracted answer: 24.6
Result:           Wrong
--------------------------------------------------


...


100%|█████████▉| 1046/1047 [11:13<00:00,  1.04it/s]


--------------------------------------------------
Question 1046 of 1047
Calculator:       Estimated Gestational Age
Output type:      integer
Ground truth:     ('34 weeks', '6 days')
Acceptable range: ('34 weeks', '6 days') - ('34 weeks', '6 days')
Model response:   {"answer": "(10 weeks, 5 days)"}
Extracted answer: (10, 5)
Result:           Wrong
--------------------------------------------------



100%|██████████| 1047/1047 [11:14<00:00,  1.55it/s]


--------------------------------------------------
Question 1047 of 1047
Calculator:       Estimated Gestational Age
Output type:      integer
Ground truth:     ('14 weeks', '2 days')
Acceptable range: ('14 weeks', '2 days') - ('14 weeks', '2 days')
Model response:   {"answer": "(5 weeks, 4 days)"}
Extracted answer: (5, 4)
Result:           Wrong
--------------------------------------------------

EVALUATION COMPLETE
Total questions:          1047
Correct answers:          190
Wrong answers:            856
Skipped/Invalid:          1
Coverage:                 99.90%
Accuracy (all 1047):   18.15%


{'lab': {'average': np.float64(15.9), 'std': np.float64(0.02)},
 'risk': {'average': np.float64(7.5), 'std': np.float64(0.02)},
 'physical': {'average': np.float64(34.58), 'std': np.float64(0.03)},
 'severity': {'average': np.float64(17.5), 'std': np.float64(0.04)},
 'diagnosis': {'average': np.float64(28.33), 'std': np.float64(0.06)},
 'date': {'average': np.float64(5.0), 'std': np.float64(0.03)},
 'dosage': {'average': np.float64(7.5), 'std': np.float64(0.04)},
 'overall': {'average': np.float64(18.15), 'std': np.float64(0.01)}}